# Train on Colab (GPU)

Runs `scripts/train.py` on a Colab GPU and saves the log, the new `results.csv` rows and the checkpoint to Google Drive (`MyDrive/tiger-reid-runs/`). Uses the same frozen `splits/splits_v1.csv` as local runs.

**One-time setup**
1. *Runtime → Change runtime type → T4 GPU*.
2. The repo is private. Create a GitHub **fine-grained token**: only `Adi-Kaul/tiger-reid`, permission *Contents: Read-only*.
3. In Colab, click the 🔑 *Secrets* icon in the left sidebar → add `GH_TOKEN` = the token → turn on notebook access.

Then *Runtime → Run all*. To run a different model, change `CONFIG` / `SEED` in the settings cell.

## Settings

In [ ]:
CONFIG = "configs/resnet50_arcface.yaml"   # resnet50_triplet.yaml, dinov2s_arcface.yaml, ...
SEED = 0
EXTRA_ARGS = ""          # e.g. "--epochs 5" for a quick test; "--eval-test" only for final configs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code + environment

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/tiger-reid"
if not os.path.exists(REPO):
    tok = userdata.get("GH_TOKEN")
    subprocess.run(["git", "clone", "-q", f"https://{tok}@github.com/Adi-Kaul/tiger-reid.git", REPO], check=True)
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/Adi-Kaul/tiger-reid.git"], check=True)
else:
    print("repo already cloned; pulling (needs token)")
    tok = userdata.get("GH_TOKEN")
    subprocess.run(["git", "-C", REPO, "pull", "-q", f"https://{tok}@github.com/Adi-Kaul/tiger-reid.git", "main"], check=True)
%cd {REPO}
!git log --oneline -1

In [ ]:
# torch / torchvision / numpy / pandas / opencv are preinstalled on Colab
!pip install -q timm pytorch-metric-learning pyyaml tabulate
!pip install -q --no-deps -e .

In [ ]:
!python scripts/download_atrw.py   # ATRW re-ID, ~220 MB, ~1 min

In [ ]:
!python -m pytest -q 2>&1 | tail -3   # should be all passed

## Train

In [ ]:
import yaml
name = yaml.safe_load(open(CONFIG))["name"]
run = f"{name}_s{SEED}"
n_rows_before = sum(1 for _ in open("results/results.csv"))
!python scripts/train.py --config {CONFIG} --seed {SEED} --num-workers 2 {EXTRA_ARGS} 2>&1 | tee run.log

## Save to Google Drive

In [ ]:
import shutil, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

out = Path(f"/content/drive/MyDrive/tiger-reid-runs/{run}_{time.strftime('%Y%m%d-%H%M')}")
out.mkdir(parents=True, exist_ok=True)
lines = open("results/results.csv").read().splitlines(keepends=True)
(out / "new_rows.csv").write_text(lines[0] + "".join(lines[n_rows_before:]))
shutil.copy("run.log", out)
shutil.copy(f"results/train_logs/{run}.csv", out / "train_log.csv")
shutil.copy(f"checkpoints/{run}.pt", out)
print("saved ->", out)
!ls -la "{out}"
print(open(out / "new_rows.csv").read())